# Deeper CNN + BatchNorm + GAP on Preprocessed FMA Medium

This notebook **does not decode MP3 files**. It expects `data_preprocessing.ipynb` to already have created:

```text
fma_preprocessed_medium/
├── spectrograms_manifest.csv
└── spectrograms/
```

It trains the best architecture from the small-dataset experiments: **Deeper CNN + BatchNorm + Global Average Pooling**.


In [1]:
# Dependencies are already installed in the cluster `fma_cnn` environment.
# Do NOT run `%pip install ...` inside the Slurm job.
print("Using existing cluster environment.")

Using existing cluster environment.


In [2]:
import os
from pathlib import Path
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

Torch: 2.12.0+cu130
CUDA available: False
CUDA device count: 0


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path(os.environ.get("FMA_PROJECT_DIR", Path.cwd())).resolve()

# Your preprocessing output is currently here on the cluster:
#   ~/fma_project/fma/fma_preprocessed_medium
default_processed = PROJECT_DIR / "fma_preprocessed_medium"
if not default_processed.exists():
    default_processed = PROJECT_DIR / "data" / "fma_preprocessed_medium"

PROCESSED_DIR = Path(os.environ.get("PROCESSED_DIR", default_processed)).resolve()
MANIFEST_PATH = Path(os.environ.get("MANIFEST_PATH", PROCESSED_DIR / "spectrograms_manifest.csv")).resolve()

RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", PROJECT_DIR / "results_medium_deeper_batchnorm_gap_preprocessed")).resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("Manifest exists:", MANIFEST_PATH.exists())

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

PROJECT_DIR: /home-mscluster/skapenga/fma_project/fma
PROCESSED_DIR: /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium
MANIFEST_PATH: /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms_manifest.csv
RESULTS_DIR: /home-mscluster/skapenga/fma_project/fma/results_medium_deeper_batchnorm_gap_preprocessed
Manifest exists: True


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Hyperparameters
# ─────────────────────────────────────────────────────────────────────────────
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "32"))
NUM_EPOCHS = int(os.environ.get("NUM_EPOCHS", "30"))
LR = float(os.environ.get("LR", "0.001"))
DROPOUT = float(os.environ.get("DROPOUT", "0.4"))
PATIENCE = int(os.environ.get("PATIENCE", "7"))
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "4"))

print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("LR:", LR)
print("DROPOUT:", DROPOUT)
print("PATIENCE:", PATIENCE)
print("NUM_WORKERS:", NUM_WORKERS)

BATCH_SIZE: 32
NUM_EPOCHS: 30
LR: 0.001
DROPOUT: 0.4
PATIENCE: 7
NUM_WORKERS: 2


## Load preprocessed manifest

In [5]:
manifest = pd.read_csv(MANIFEST_PATH)
print("Manifest shape:", manifest.shape)
print("Columns:", list(manifest.columns))
display(manifest.head())

# Normalise expected column names.
# The preprocessing notebook should include: spectrogram_path, split, genre, label.
if "spectrogram_path" not in manifest.columns:
    possible_path_cols = [c for c in manifest.columns if "spectrogram" in c.lower() or "path" in c.lower()]
    raise ValueError(f"No 'spectrogram_path' column found. Possible path columns: {possible_path_cols}")

if "split" not in manifest.columns:
    possible_split_cols = [c for c in manifest.columns if "split" in c.lower()]
    raise ValueError(f"No 'split' column found. Possible split columns: {possible_split_cols}")

# Make spectrogram paths absolute and robust.
def resolve_spec_path(p):
    p = Path(str(p))
    if p.is_absolute():
        return str(p)
    # Most preprocessing notebooks save paths relative to PROCESSED_DIR or project root.
    candidate1 = PROCESSED_DIR / p
    candidate2 = PROJECT_DIR / p
    if candidate1.exists():
        return str(candidate1)
    if candidate2.exists():
        return str(candidate2)
    return str(candidate1)

manifest["spectrogram_path"] = manifest["spectrogram_path"].apply(resolve_spec_path)

# Create/confirm labels.
if "label" not in manifest.columns:
    genre_col = "genre" if "genre" in manifest.columns else None
    if genre_col is None:
        possible_genre_cols = [c for c in manifest.columns if "genre" in c.lower()]
        raise ValueError(f"No label column and no genre column found. Possible genre columns: {possible_genre_cols}")
    class_names = sorted(manifest[genre_col].dropna().unique().tolist())
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    manifest["label"] = manifest[genre_col].map(class_to_idx).astype(int)
else:
    manifest["label"] = manifest["label"].astype(int)
    if "genre" in manifest.columns:
        class_map = manifest[["label", "genre"]].drop_duplicates().sort_values("label")
        class_names = class_map.groupby("label")["genre"].first().tolist()
    else:
        class_names = [str(i) for i in sorted(manifest["label"].unique())]

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)
print("Class names:", class_names)
print("Split counts:")
print(manifest["split"].value_counts())
print("Label counts:")
print(manifest["label"].value_counts().sort_index())

# Check a few files exist.
sample_paths = manifest["spectrogram_path"].head(5).tolist()
for p in sample_paths:
    print(p, "exists:", Path(p).exists())

missing = manifest[~manifest["spectrogram_path"].apply(lambda x: Path(x).exists())]
print("Missing spectrogram files:", len(missing))
if len(missing) > 0:
    raise FileNotFoundError(f"{len(missing)} spectrogram files are missing. First missing: {missing.iloc[0]['spectrogram_path']}")

with open(RESULTS_DIR / "class_names.txt", "w") as f:
    for c in class_names:
        f.write(str(c) + "\n")

Manifest shape: (24980, 10)
Columns: ['track_id', 'split', 'genre', 'label', 'spectrogram_path', 'audio_path', 'duration', 'bit_rate', 'title', 'artist']


,track_id,split,genre,label,spectrogram_path,audio_path,duration,bit_rate,title,artist
0,2,training,Hip-Hop,7,/home-mscluster/skapenga/fma_project/fma/fma_p...,/datasets/skapenga/kaspoas/fma_medium/000/0000...,168,256000,Food,AWOL
1,3,training,Hip-Hop,7,/home-mscluster/skapenga/fma_project/fma/fma_p...,/datasets/skapenga/kaspoas/fma_medium/000/0000...,237,256000,Electric Ave,AWOL
2,5,training,Hip-Hop,7,/home-mscluster/skapenga/fma_project/fma/fma_p...,/datasets/skapenga/kaspoas/fma_medium/000/0000...,206,256000,This World,AWOL
3,10,training,Pop,12,/home-mscluster/skapenga/fma_project/fma/fma_p...,/datasets/skapenga/kaspoas/fma_medium/000/0000...,161,192000,Freeway,Kurt Vile
4,134,training,Hip-Hop,7,/home-mscluster/skapenga/fma_project/fma/fma_p...,/datasets/skapenga/kaspoas/fma_medium/000/0001...,207,256000,Street Music,AWOL


Number of classes: 16
Class names: ['Blues', 'Classical', 'Country', 'Easy Listening', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Jazz', 'Old-Time / Historic', 'Pop', 'Rock', 'Soul-RnB', 'Spoken']
Split counts:
split
training      19904
test           2572
validation     2504
Name: count, dtype: int64
Label counts:
label
0       74
1      619
2      178
3       21
4     6311
5     2250
6     1518
7     2192
8     1349
9     1018
10     384
11     510
12    1186
13    7098
14     154
15     118
Name: count, dtype: int64
/home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms/000/000002.npy exists: True
/home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms/000/000003.npy exists: True
/home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms/000/000005.npy exists: True
/home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms/000/000010.npy exists: True
/home-msclu

Missing spectrogram files: 0


## Dataset and DataLoaders

In [6]:
class FMASpectrogramDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        spec = np.load(row["spectrogram_path"]).astype(np.float32)

        # Expected common shapes:
        #   (64, T)       -> add channel -> (1, 64, T)
        #   (1, 64, T)    -> already correct
        #   (T, 64)       -> transpose if needed
        if spec.ndim == 2:
            # If frequency dimension appears second, transpose to (freq, time).
            if spec.shape[0] > spec.shape[1]:
                spec = spec.T
            spec = np.expand_dims(spec, axis=0)

        if spec.ndim != 3:
            raise ValueError(f"Unexpected spectrogram shape {spec.shape} for {row['spectrogram_path']}")

        x = torch.from_numpy(spec)
        y = torch.tensor(int(row["label"]), dtype=torch.long)
        return x, y


train_df = manifest[manifest["split"] == "training"].copy()
val_df = manifest[manifest["split"] == "validation"].copy()
test_df = manifest[manifest["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_dataset = FMASpectrogramDataset(train_df)
val_dataset = FMASpectrogramDataset(val_df)
test_dataset = FMASpectrogramDataset(test_df)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)

xb, yb = next(iter(train_loader))
print("Batch X shape:", tuple(xb.shape))
print("Batch y shape:", tuple(yb.shape))
print("Device:", device)

Train: (19904, 10)
Validation: (2504, 10)
Test: (2572, 10)


Batch X shape: (32, 1, 64, 1001)
Batch y shape: (32,)
Device: cpu


## Model: Deeper CNN + BatchNorm + Global Average Pooling

In [7]:
class ConvBNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

    def forward(self, x):
        return self.block(x)


class DeeperBatchNormGAPCNN(nn.Module):
    def __init__(self, num_classes: int, dropout: float = 0.4):
        super().__init__()

        self.features = nn.Sequential(
            ConvBNBlock(1, 32),
            ConvBNBlock(32, 64),
            ConvBNBlock(64, 128),
            ConvBNBlock(128, 256),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        return self.classifier(x)


model = DeeperBatchNormGAPCNN(num_classes=NUM_CLASSES, dropout=DROPOUT).to(device)

if device.type == "cuda" and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

print(model)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count:,}")

with open(RESULTS_DIR / "model_summary.txt", "w") as f:
    f.write(str(model))
    f.write("\n")
    f.write(f"Model parameters: {param_count:,}\n")

DeeperBatchNormGAPCNN(
  (features): Sequential(
    (0): ConvBNBlock(
      (block): Sequential(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBNBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBNBlock(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_runnin

## Training and evaluation helpers

In [8]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in tqdm(loader, desc="Training", leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for xb, yb in tqdm(loader, desc=desc, leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(yb.cpu().numpy().tolist())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def plot_history(history_df):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_accuracy"], label="Training Accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("FMA Medium Deeper CNN: Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(RESULTS_DIR / "accuracy.png", dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Training Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("FMA Medium Deeper CNN: Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(RESULTS_DIR / "loss.png", dpi=300, bbox_inches="tight")
    plt.close()

## Train model

In [9]:
# Class weights help because FMA medium is unbalanced.
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history = []
best_val_loss = float("inf")
best_val_acc = 0.0
epochs_without_improvement = 0

best_model_path = RESULTS_DIR / "best_model.pt"
final_model_path = RESULTS_DIR / "final_model.pt"

print("Starting training:", datetime.now())

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device, desc="Validation")

    scheduler.step(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })

    print(f"Train:      loss={train_loss:.4f} acc={train_acc:.4f}")
    print(f"Validation: loss={val_loss:.4f} acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        epochs_without_improvement = 0
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state, best_model_path)
        print(f"  -> best model saved: val_loss={best_val_loss:.4f}, val_acc={best_val_acc:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"  -> no improvement for {epochs_without_improvement}/{PATIENCE} epochs")

    history_df = pd.DataFrame(history)
    history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

# Save final state too.
state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save(state, final_model_path)

history_df = pd.DataFrame(history)
plot_history(history_df)
print("Training finished:", datetime.now())

Starting training: 2026-05-22 12:26:28.013548

Epoch 1/30


Training:   0%|          | 0/622 [00:00<?, ?it/s]

Validation:   0%|          | 0/79 [00:00<?, ?it/s]

Train:      loss=2.3164 acc=0.3176
Validation: loss=2.0725 acc=0.3934
  -> best model saved: val_loss=2.0725, val_acc=0.3934

Epoch 2/30


Training:   0%|          | 0/622 [00:00<?, ?it/s]

...

Train:      loss=1.2656 acc=0.5600
Validation: loss=1.6048 acc=0.5723
  -> no improvement for 6/7 epochs

Epoch 29/30


Training:   0%|          | 0/622 [00:00<?, ?it/s]

Validation:   0%|          | 0/79 [00:00<?, ?it/s]

Train:      loss=1.2549 acc=0.5649
Validation: loss=1.6711 acc=0.5867
  -> no improvement for 7/7 epochs
Early stopping triggered.


Training finished: 2026-05-22 15:42:12.560915


## Test evaluation

In [10]:
# Load best model before testing.
base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.load_state_dict(torch.load(best_model_path, map_location=device))

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device, desc="Testing")

print(f"Test loss={test_loss:.4f} acc={test_acc:.4f}")

report = classification_report(test_labels, test_preds, target_names=class_names, zero_division=0)
print(report)

with open(RESULTS_DIR / "classification_report.txt", "w") as f:
    f.write(report)

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(13, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=60, cmap="Blues", colorbar=True)
plt.title("FMA Medium Deeper CNN: Confusion Matrix")
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.close()

weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="weighted", zero_division=0
)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="macro", zero_division=0
)

results = {
    "name": "fma_medium_deeper_batchnorm_gap_preprocessed",
    "actual_epochs": int(len(history_df)),
    "best_val_accuracy": float(history_df["val_accuracy"].max()),
    "best_val_loss": float(history_df["val_loss"].min()),
    "final_train_accuracy": float(history_df["train_accuracy"].iloc[-1]),
    "final_val_accuracy": float(history_df["val_accuracy"].iloc[-1]),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "weighted_precision": float(weighted_precision),
    "weighted_recall": float(weighted_recall),
    "weighted_f1": float(weighted_f1),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "num_classes": int(NUM_CLASSES),
    "classes": list(map(str, class_names)),
    "batch_size": int(BATCH_SIZE),
    "epochs_requested": int(NUM_EPOCHS),
    "patience": int(PATIENCE),
    "learning_rate": float(LR),
    "dropout": float(DROPOUT),
}

with open(RESULTS_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=4)

pd.DataFrame([results]).to_csv(RESULTS_DIR / "medium_deeper_summary.csv", index=False)

print(json.dumps(results, indent=4))
print("Results saved to:", RESULTS_DIR)

Testing:   0%|          | 0/81 [00:00<?, ?it/s]

Test loss=1.7491 acc=0.5226
                     precision    recall  f1-score   support

              Blues       0.13      0.25      0.17         8
          Classical       0.46      0.87      0.60        62
            Country       0.02      0.11      0.03        18
     Easy Listening       0.00      0.00      0.00         6
         Electronic       0.80      0.57      0.67       632
       Experimental       0.33      0.32      0.33       225
               Folk       0.28      0.30      0.29       152
            Hip-Hop       0.58      0.77      0.66       220
       Instrumental       0.23      0.31      0.26       174
      International       0.41      0.29      0.34       102
               Jazz       0.34      0.54      0.42        39
Old-Time / Historic       0.93      1.00      0.96        51
                Pop       0.20      0.25      0.22       119
               Rock       0.81      0.61      0.70       710
           Soul-RnB       0.09      0.12      0.10      

{
    "name": "fma_medium_deeper_batchnorm_gap_preprocessed",
    "actual_epochs": 29,
    "best_val_accuracy": 0.5866613418530351,
    "best_val_loss": 1.56092323853185,
    "final_train_accuracy": 0.5648613344051447,
    "final_val_accuracy": 0.5866613418530351,
    "test_loss": 1.7490959909062362,
    "test_accuracy": 0.5225505443234837,
    "weighted_precision": 0.5963417543198212,
    "weighted_recall": 0.5225505443234837,
    "weighted_f1": 0.5458369306651943,
    "macro_precision": 0.3639884187951964,
    "macro_recall": 0.4420864265933116,
    "macro_f1": 0.38045438620011474,
    "num_classes": 16,
    "classes": [
        "Blues",
        "Classical",
        "Country",
        "Easy Listening",
        "Electronic",
        "Experimental",
        "Folk",
        "Hip-Hop",
        "Instrumental",
        "International",
        "Jazz",
        "Old-Time / Historic",
        "Pop",
        "Rock",
        "Soul-RnB",
        "Spoken"
    ],
    "batch_size": 32,
    "epochs_